# Notebook 5: Interactive visualization of subgraphs

Import the Python library dependencies.

In [ ]:
import sys

import watermark

import kuzu
#import ryugraph

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions

## yFiles Interactive Visualization

Reconnect to the existing graph database.

In [ ]:
DB_PATH: str = "./db"

if "kuzu" in sys.modules:
    db: kuzu.Database = kuzu.Database(DB_PATH)
    conn: kuzu.Connection = kuzu.Connection(db)

    from yfiles_jupyter_graphs_for_kuzu import KuzuGraphWidget
    widget: KuzuGraphWidget = KuzuGraphWidget(conn)
else:
    db: ryugraph.Database = ryugraph.Database(DB_PATH)
    conn: ryugraph.Connection = ryugraph.Connection(db)

    from yfiles_jupyter_graphs_for_ryugraph import RyuGraphWidget
    widget: RyuGraphWidget = RyuGraphWidget(conn)

Define a function to adjust the node and edge configurations, to help clarify the visualization.
This uses [Viz Palette](https://projects.susielu.com/viz-palette).

In [ ]:
def node_color (
    node: dict
    ) -> str:
    match node["properties"]["class"]:
        case "sz:Person":
            return "#B8C25F"
        case "sz:Organization":
            return "#8C6894"
        case "sz:DataRecord":
            return "#B7B38E"
        case _:
            return "#000000"

Define a function to customize the node scaling -- within a range of `(0.0, 2.0]` -- based on the computed centrality measures.

In [ ]:
def node_scale_factor (
    node: dict,
    base: float = 0.8,
    rate: float = 20.0,
    ) -> float:
    centrality: float = node["properties"]["betweenness_centrality"]
    limit: float = 2.0 - base
    curve: float = limit * (rate**centrality - 1.0) / max(0.01, (rate - 1.0))
    
    return min(
        2.0,
        curve + base,
    )

Now configure a [`yFiles`](https://www.yworks.com/products/yfiles-graphs-for-jupyter) graph widget to explore the subgraphs.
Design for nodes:

In [ ]:
widget.add_node_configuration(
    "Entity",
    color = node_color,
    text = lambda node: {"text": node["properties"]["descrip"]},
    scale_factor = node_scale_factor,
)

widget.add_node_configuration(
    "OpenSanctions",
    color = "#D9CAD7",
    text = lambda node: {"text": node["properties"]["descrip"]},
    scale_factor = node_scale_factor,
)

widget.add_node_configuration(
    "OpenOwnership",
    color = "#B8C25F",
    text = lambda node: {"text": node["properties"]["descrip"]},
    scale_factor = node_scale_factor,
)

widget.add_node_configuration(
    "Risk",
    color = "#C25FB8",
    text = lambda node: {"text": node["properties"]["topic"]},
    size = (23, 23),
)



Design for edges:

In [ ]:
widget.add_relationship_configuration(
    "Related",
    color = "#949068",
    text = lambda edge: {"text": edge["properties"]["sem_rel"]},
)

widget.add_relationship_configuration(
    "Matched",
    color = "#D9CAD7",
    text = lambda edge: {"text": edge["properties"]["sem_rel"]},
)

widget.add_relationship_configuration(
    "Role",
    color = "#C25FB8",
    text = lambda edge: {"text": edge["properties"]["role"]},
)

widget.add_relationship_configuration(
    "HasRisk",
    color = "#C25FB8",
)

Run an interactive visualization using `yFiles` to examine the subgraph describing the [2021 South London Papa Johns](https://www.newsshopper.co.uk/news/19164815.boss-bromley-catford-papa-johns-stores-jailed/) tax evasion case.

As you scroll over different graph elements, notice how the design of the interactive visualization highlights important details for investigators to notice:

 - tool-tips show the properties of nodes and edges, i.e., information collected into the KG
 - relative node size indicates the scaled _betweenness centrality_ measures
 - the `evidence` property on edges shows the _entity resolution_ matching criteria, which includes _ultimate benefial ownership_ (UBO) information from Open Ownership
 - the `Risk` nodes show known risks from OpenSanctions about the entities to which they are connected

In [ ]:
query: str = """
MATCH (a)-[b]->(c:Entity)-[d *1..5]->(e)
WHERE c.descrip CONTAINS "Abassin"
RETURN * LIMIT 200;
"""

widget.show_cypher(
    query,
    layout = "radial"
)

In particular, notice how [Abassin Badshah](https://qsrmedia.co.uk/legal/more-news/papa-johns-franchisee-jailed-tax-fraud-report) is the most prominant node in this subgraph, with beneficial ownership of four shell companies involved in the tax evasion case, plus a risk noted from [UK Companies House](https://find-and-update.company-information.service.gov.uk/disqualified-officers/natural/mGquuTbmESWiRmHJPz1ObUwfDgk) for _corporate disqualification_.

Let's see how the same results look as a dataframe.

In [ ]:
conn.execute(query).get_as_pl()

## Design for Human Scale

Consider the side-by-side comparison of using of an interactive visualization versus using a table of query results (admittedly the table could be formatted much better!)

The effectiveness of using visualization builds on the results of graph algorithms, such as partitioning and centrality measures to parameterize the design, plus using _risk_ and _UBO_ information connected into the graph through _entity resolution_. This is a realization of the **four-step design pattern** mentioned earlier.

Rather than trying to visualize "billions and billions" of elements in some ginormous framework, people using these kinds of investigative graphs need to focus on smaller subgraphs. For example, the 2021 case examined above has less than a dozen nodes. This is true whether the investigation is about [_hunting bad guys_](https://en.wikipedia.org/wiki/Bringing_Down_the_House_(book)) or about trying to find your best customers within complex data.

Investigative graphs in industry and government tend to rely on people using _case management tools_, so we do this work to reshape subgraphs of potential interest, running each through case management. Moreover, it's clear that the important tools in this process are _entity resolution_, _computable semantics_, _graph analytics_, and especially the use of effective **design for human scale**. 

This latter point -- about design, UX, UI -- has become a bottleneck for the adoption of AI applications in production use. However, the converse is where we leverage AI technologies to "refocus the lens" so that investigative teams can collaborate more effectively. Whereas the criminal tradecraft plus the enduring problems of data quality in enterprise both cause more [_complexity_](https://cynefin.io/wiki/Field_guide_to_managing_complexity_(and_chaos)_in_times_of_crisis) in these kinds of use cases, here in this tutorial we're showing production-quality approaches for "refocusing the lens". In other words, the intent is to bring inhumanly large, complex problems back to _human scale_. This is largely a design problem, which is becoming an AI problem.

Some recommended resources about **Design for Human Scale**:

_Design for Human Scale_  
**Victor Papanek**  
(1981)  
<https://www.goodreads.com/en/book/show/2845800-design-for-human-scale>

_Humanscale_  
**Alvin Tilley**, **Joan Bardagjy**  
(1991)  
<https://www.goodreads.com/book/show/1725112.Humanscale>

_Understanding Computers and Cognition: A New Foundation for Design_  
**Terry Winograd**, **Fernando Flores**  
(1986)  
<https://www.goodreads.com/book/show/53482.Understanding_Computers_and_Cognition>

_Conversations For Action and Collected Essays: Instilling a Culture of Commitment in Working Relationships_  
**Fernando Flores**, **Maria Flores Letelier**  
(2013)  
<https://www.goodreads.com/book/show/17870114-conversations-for-action-and-collected-essays>

> _Author Note:_ as grad student I focused on AI though also took an extra year to participate in the Design program at Stanford. The intent was to explore ways of blending **Artificial Intelligence** technologies plus **Design for Human Scale** together. _Mas o menos_, this now seems to be turning into almost a thesis statement.

> Also, thank you for participating in this course.

---

Finally, close the database connection.

In [ ]:
db.close()

---